# ЛР2: проверка научных утверждений с RAG

Код: Грязнов Александр Евгеньевич, 8ВМ63. Подготовлен с ИИ-помощником.

Локально уже сохранены 420 заданий: LLM-only 90, RAG k=1/3/5 по 90, oracle 60.
Здесь используем одну T4, Qwen3-4B-Instruct-2507 FP16 и vLLM 0.33.0.

**До запуска:** добавь приватный Dataset с `lab2_inputs_95219452.zip`, включи Internet и выбери GPU T4.
Если Kaggle выделяет T4 ×2, процессы увидят только GPU 0.

Ячейки выполняются по порядку. При ошибке дальнейшие этапы не запускай.
Веса и окружение находятся в `/tmp`, результаты — в `/kaggle/working/lab2_artifacts`.
Последовательность пакетов: проверка, эксперимент 1, промежуточный архив, эксперимент 2, итоговый архив.
Таблицы качества и MLflow подготовим локально после скачивания результатов.


## 1. Настройки и проверка GPU

В PowerShell после коммита и `git push` выполни `git rev-parse HEAD`.
Вставь полученные **40 символов** в поле ввода ниже. Это версия кода запуска.
Версия подготовки входов отдельно сохранена внутри архива.


In [ ]:
import os
import sys
import json
import shutil
import hashlib
import subprocess
import zipfile
from pathlib import Path

ART = Path("/kaggle/working/lab2_artifacts")
INPUTS = ART / "inputs"
REPO = Path("/tmp/intsys-labs-lab2")
PYTHON = "/tmp/lab2-env/bin/python"
ART.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ)
ENV.pop("PYTHONPATH", None)
ENV.update(CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/lab2-hf",
           PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1",
           OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg",
           LAB2_ARTIFACTS=str(ART))

def cmd(arguments, cwd=None, show=True):
    process = subprocess.Popen(arguments, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        lines.append(line)
        if show:
            print(line, end="", flush=True)
    code = process.wait()
    text = "".join(lines)
    if code:
        if not show:
            print(text)
        raise RuntimeError(f"Команда завершилась с кодом {code}: {arguments}")
    return text

COMMIT = input("Вставь полный хеш из git rev-parse HEAD: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT), "Нужны 40 шестнадцатеричных символов"
print("Коммит запуска:", COMMIT)
print("Python блокнота:", sys.version)
cmd(["nvidia-smi"])
gpu = cmd(["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"], show=False).strip()
assert "T4" in gpu, f"Ожидали T4, получено: {gpu}"
print("Используем только GPU 0:", gpu)


## 2. Восстановление подготовленных входов

Проверяем исходный ZIP и хеши файлов. Если Kaggle автоматически распаковал ZIP при создании Dataset,
найдём развёрнутые файлы и проверим их по закреплённому хешу метаданных.
Эта ячейка не пересчитывает поиск и не изменяет промпты.


In [ ]:
EXPECTED_ID = "95219452bef84676a0932e060d22b893"
EXPECTED_ZIP_SHA = "c094a4dd19f02b92c501871b68fc9574668bcc596775852aa200daf0e1304139"
EXPECTED_META_SHA = "c0c72a0d8f1397881c95993031aa6673582657ff5e73a7d902c79d9e9a478457"
NAME = "lab2_inputs_95219452.zip"

def digest(data):
    return hashlib.sha256(data).hexdigest()

zips = sorted(Path("/kaggle/input").rglob(NAME))
verified_zip = None
if zips:
    source = zips[0]
    assert digest(source.read_bytes()) == EXPECTED_ZIP_SHA, "SHA-256 ZIP не совпал"
    verified_zip = EXPECTED_ZIP_SHA
    with zipfile.ZipFile(source) as archive:
        assert archive.testzip() is None, "ZIP повреждён"
        metadata_bytes = archive.read("metadata.json")
        assert digest(metadata_bytes) == EXPECTED_META_SHA, "Метаданные не совпали"
        preparation = json.loads(metadata_bytes)
        required = ["metadata.json", *preparation["files_sha256"],
                    *("provenance/" + name for name in preparation["source_sha256"])]
        files = {name: archive.read(name) for name in required}
else:
    matches = [p for p in Path("/kaggle/input").rglob("metadata.json")
               if digest(p.read_bytes()) == EXPECTED_META_SHA]
    assert matches, "Добавь Dataset с правильным ZIP или его распакованными файлами"
    source = matches[0].parent
    metadata_bytes = (source / "metadata.json").read_bytes()
    preparation = json.loads(metadata_bytes)
    required = ["metadata.json", *preparation["files_sha256"],
                *("provenance/" + name for name in preparation["source_sha256"])]
    files = {name: (source / name).read_bytes() for name in required}
    print("Kaggle распаковал ZIP: проверяем метаданные и содержимое.")

assert preparation["run_id"] == EXPECTED_ID and not preparation["debug"]
assert preparation["status"] == "completed" and preparation["task_count"] == 420
for name, expected in preparation["files_sha256"].items():
    assert digest(files[name]) == expected, f"Повреждён файл {name}"
for name, expected in preparation["source_sha256"].items():
    assert digest(files["provenance/" + name]) == expected, f"Повреждён снимок {name}"
for name, data in files.items():
    target = INPUTS / name
    assert target.resolve().is_relative_to(INPUTS.resolve())
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert target.read_bytes() == data, f"Существующий файл отличается: {target}"
    else:
        target.write_bytes(data)
(ART / "input_transfer.json").write_text(json.dumps({
    "preparation_run_id": EXPECTED_ID, "metadata_sha256": EXPECTED_META_SHA,
    "original_zip_sha256_expected": EXPECTED_ZIP_SHA,
    "original_zip_sha256_verified": verified_zip, "source": str(source),
}, indent=2), encoding="utf-8")
print("Входы проверены:", INPUTS)
print("Подготовлено заданий:", preparation["task_count"])


## 3. Получение зафиксированной версии кода

Клонируем GitHub и переключаемся на введённый коммит. Исходники, которыми готовились входы,
сверяются со снимками в архиве. Разница CRLF/LF при переносе Windows → Linux допускается.


In [ ]:
if not REPO.exists():
    cmd(["git", "clone", "https://github.com/manaer6-alt/intsys-labs.git", str(REPO)])
cmd(["git", "-C", str(REPO), "fetch", "origin", COMMIT])
cmd(["git", "-C", str(REPO), "checkout", "--detach", COMMIT])
assert cmd(["git", "-C", str(REPO), "rev-parse", "HEAD"], show=False).strip() == COMMIT
assert not cmd(["git", "-C", str(REPO), "status", "--porcelain"], show=False).strip()
for name in preparation["source_sha256"]:
    current = (REPO / name).read_bytes().replace(b"\r\n", b"\n")
    old = files["provenance/" + name].replace(b"\r\n", b"\n")
    assert current == old, f"Исходник изменён после подготовки: {name}"
assert (REPO / "lab2/run_inference.py").exists(), "В коммите отсутствует клиент ЛР2"
print("Код и входы согласованы.")


## 4. Отдельное окружение и проверка CUDA

Воспроизводим вариант установки из ЛР1: vLLM 0.33.0 и Python 3.13.
Ядро блокнота управляет процессами; вычислительные библиотеки запускаются Python из `/tmp/lab2-env`.
Перезапуск ядра после этой установки не требуется.


In [ ]:
UV = shutil.which("uv")
if UV is None:
    cmd([sys.executable, "-m", "pip", "install", "uv"])
    UV = shutil.which("uv")
UV_CMD = [UV] if UV else [sys.executable, "-m", "uv"]
if not Path(PYTHON).exists():
    cmd(UV_CMD + ["venv", "--python", "3.13", "--seed", "/tmp/lab2-env"])
cmd(UV_CMD + ["pip", "install", "--python", PYTHON,
              "-r", str(REPO / "lab2/requirements-gpu.txt"), "--torch-backend=auto"])
cmd(UV_CMD + ["pip", "check", "--python", PYTHON])
freeze = cmd(UV_CMD + ["pip", "freeze", "--python", PYTHON], show=False)
(ART / "requirements_gpu.txt").write_text(freeze, encoding="utf-8")
(ART / "uv_version.txt").write_text(cmd(UV_CMD + ["--version"], show=False), encoding="utf-8")

check_code = r'''
import json, os, platform
from pathlib import Path
from importlib.metadata import version
import torch

assert torch.cuda.is_available(), "CUDA недоступна"
assert torch.cuda.device_count() == 1, "Процесс должен видеть только одну GPU"
assert "T4" in torch.cuda.get_device_name(0), "Нужна T4"
torch.set_num_threads(1)
value = torch.ones(1, device="cuda").cpu().item()
record = {
    "python": platform.python_version(),
    "versions": {name: version(name) for name in (
        "vllm", "torch", "transformers", "huggingface-hub", "tokenizers", "numpy", "openai", "PyYAML", "wrapt"
    )},
    "torch_cuda": torch.version.cuda, "cuda_device_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "cuda_test_value": value, "cpu_logical_count": os.cpu_count(),
    "environment_variables": {k: os.environ.get(k) for k in (
        "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS", "TOKENIZERS_PARALLELISM"
    )},
}
(Path(os.environ["LAB2_ARTIFACTS"]) / "environment.json").write_text(
    json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(record, ensure_ascii=False, indent=2))
'''
cmd([PYTHON, "-c", check_code])


## 5. Запуск Qwen

Загружаем ту же ревизию Qwen, что в подготовленных входах. Одна T4, FP16,
предел контекста 32768, кеширование префикса выключено. Полная команда и журнал сохраняются.


In [ ]:
sys.path.insert(0, str(REPO / "lab2"))
from kaggle_control import KaggleRunner

runner = KaggleRunner(INPUTS, ART, PYTHON, ENV)
runner.start_server()


## 6. Проверка перед основными сериями

Сверяем через `/tokenize` 420 основных входов и два входа прогрева. Это подсчёт токенов, без генерации.
Затем получаем два пробных JSON-ответа на claims_train и проверяем usage.
При несовпадении длин основные серии блокируются.


In [ ]:
runner.stage("check")
check_archive = runner.checkpoint("check")


## 7. Эксперимент 1: без поиска, RAG k=3, oracle

Выполняем 90 + 90 + 60 основных вызовов. Каждый режим — отдельный процесс и run_id.
Прогрев не входит в основные результаты. После трёх режимов сохраняем промежуточный архив.


In [ ]:
for stage in ("llm_only", "rag_k3", "oracle"):
    runner.stage(stage)
exp1_archive = runner.checkpoint("exp1")


## 8. Эксперимент 2: RAG k=1 и k=5

Ещё 180 основных вызовов. Результаты k=3 уже получены и повторно не запрашиваются.
После выполнения останавливаем процесс модели и создаём итоговый ZIP.


In [ ]:
for stage in ("rag_k1", "rag_k5"):
    runner.stage(stage)
runner.stop_server()
final_archive = runner.checkpoint("final")
print("Скачай из Output:", final_archive.name)
print("И контрольную сумму:", final_archive.name + ".sha256")


## После выполнения

1. Скачай итоговый ZIP и `.sha256` из `/kaggle/working` через панель Output.
2. Сохрани выполненный блокнот IPYNB с выводом ячеек на ноутбук в папку lab2.
3. Пришли сводку `ИТОГИ ВЫПОЛНЕНИЯ`, имя ZIP и его SHA-256.
4. Заверши GPU-сессию Kaggle: остановка процесса модели сама по себе не отключает ускоритель.

Результаты ещё не содержат Accuracy: эталонные метки не передавались обычному клиенту генерации.
После скачивания локально пересчитаем качество, попарные переходы, таблицы и разбор ошибок, затем зарегистрируем запуски в MLflow.

При остановке ячейки контроллер сохраняет промежуточный архив. Повтор того же этапа продолжает отсутствующие записи;
сохранённые ошибки остаются в результатах. Если код или условия изменились, автоматическое продолжение блокируется.
